# Detection statistics

Statistics of the person boxes found by **stage 1 (detection)** for one video, to choose the
**Stage 1: per-frame filter** parameters (see `research/detection-filtering.md`).

**How to use:** set `DETECTIONS_PATH` below to any `data/output/<video>/1_detections.json` and run all cells.

**Note:** the file only contains detections **above the detection threshold** used for that run
(shown below, from `config.json`). Lower scores were never saved. To see them, re-run detection with a
lower threshold, e.g. `python -m swim.run --input <video> --detection-threshold 0.1`.

In [ ]:
import json
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, FuncFormatter, MaxNLocator, NullLocator
import numpy as np
import pandas as pd

# Find the repo root (the folder that contains swim/) so `import swim` works from notebooks/
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "swim").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from swim.stages.detection.types import VideoDetections

# Chart style: one series color, recessive hairline grid and axes, text in ink colors
SERIES = "#2a78d6"  # blue: the data
SURFACE = "#fcfcfb"
INK, INK_2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, AXIS = "#e1e0d9", "#c3c2b7"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.family": "sans-serif", "font.size": 10,
    "text.color": INK, "axes.labelcolor": INK_2, "axes.titlecolor": INK,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.edgecolor": AXIS, "axes.linewidth": 1, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 1, "grid.linestyle": "-", "axes.axisbelow": True,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelcolor": INK_2, "ytick.labelcolor": INK_2,
    "legend.frameon": False,
})
pd.set_option("display.width", 140)


def log_ticks(axis, ticks) -> None:
    """Plain numbers (0.5, 1, 2) instead of powers of ten on a log-scale axis."""
    axis.set_major_locator(FixedLocator(ticks))
    axis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
    axis.set_minor_locator(NullLocator())


AREA_TICKS = [0.1, 0.2, 0.5, 1, 2, 5, 10, 20, 50]  # % of frame
PX_TICKS = [20, 50, 100, 200, 500, 1000, 2000]
ASPECT_TICKS = [0.25, 0.5, 1, 2, 4, 8]

In [ ]:
# Detections to analyze: change this to any data/output/<video>/1_detections.json
DETECTIONS_PATH = REPO_ROOT / "data/output/IMG_4887/1_detections.json"

## Load

In [ ]:
video_dets = VideoDetections.from_json(DETECTIONS_PATH)
W, H = video_dets.width, video_dets.height

config_path = DETECTIONS_PATH.parent / "config.json"
DETECTION_THRESHOLD = json.loads(config_path.read_text())["detection"]["threshold"] if config_path.exists() else None

VIDEO_PATH = Path(video_dets.video_path)
if not VIDEO_PATH.is_absolute():
    VIDEO_PATH = REPO_ROOT / VIDEO_PATH

EDGE_MARGIN_PX = 2  # a box closer than this to the frame border counts as "touching the edge"

rows = []
for f in video_dets.frames:
    boxes = [d.bbox for d in f.detections]
    for i, d in enumerate(f.detections):
        b = d.bbox
        rows.append({
            "frame": f.frame_idx,
            "time_s": f.timestamp,
            "score": d.score,
            "x1": b.x1, "y1": b.y1, "x2": b.x2, "y2": b.y2,
            "width_px": b.width,
            "height_px": b.height,
            "area_pct": 100 * b.area / (W * H),  # box area as % of the frame
            "aspect": b.width / max(b.height, 1e-6),  # > 1 wide (swimming), < 1 tall (standing)
            "cx": b.center[0] / W,  # box center, 0..1 across the frame
            "cy": b.center[1] / H,  # box center, 0..1 down the frame
            "touches_edge": b.x1 <= EDGE_MARGIN_PX or b.y1 <= EDGE_MARGIN_PX
                            or b.x2 >= W - EDGE_MARGIN_PX or b.y2 >= H - EDGE_MARGIN_PX,
            "persons_in_frame": len(f.detections),
            "max_iou": max((b.iou(o) for j, o in enumerate(boxes) if j != i), default=0.0),  # overlap with another box
        })
df = pd.DataFrame(rows)

print(f"Detections: {DETECTIONS_PATH}")
print(f"Video:      {VIDEO_PATH}  ({W}x{H}, {video_dets.fps:.2f} fps)")
print(f"Model:      {video_dets.model}, detection threshold {DETECTION_THRESHOLD}")

## Overview

In [ ]:
n_frames = len(video_dets.frames)
counts = pd.Series([len(f.detections) for f in video_dets.frames])
print(f"Frames: {n_frames}   with a person: {(counts > 0).sum()} ({100 * (counts > 0).mean():.0f}%)   "
      f"detections: {len(df)}   touching the frame edge: {df.touches_edge.sum()}")
print()
print("Frames by number of persons detected:")
print(counts.value_counts().sort_index().rename_axis("persons").to_frame("frames").T.to_string())
print()
df[["score", "width_px", "height_px", "area_pct", "aspect", "max_iou"]].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
).round(3)

## Confidence scores

Everything left of the detection threshold was never saved (see the note at the top).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.2))
low = DETECTION_THRESHOLD if DETECTION_THRESHOLD is not None else df.score.min()
ax.hist(df.score, bins=np.arange(np.floor(low * 50) / 50, 1.0001, 0.02), color=SERIES, edgecolor=SURFACE, linewidth=1)
if DETECTION_THRESHOLD is not None:
    ax.axvline(DETECTION_THRESHOLD, color=MUTED, linewidth=1, linestyle="--")
    ax.annotate(f"detection threshold {DETECTION_THRESHOLD}", (DETECTION_THRESHOLD, 1), xycoords=("data", "axes fraction"),
                xytext=(4, -2), textcoords="offset points", va="top", color=INK_2, fontsize=9)
ax.set(title="Confidence score of each detection", xlabel="score", ylabel="detections")
plt.show()
print("Score percentiles:", df.score.quantile([0.01, 0.05, 0.1, 0.25, 0.5]).round(3).to_dict())

## Box size

Area is shown as **% of the frame**, on a log scale (sizes span orders of magnitude).
Tiny boxes are often splashes or reflections; huge ones a person close to the camera.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))
bins = np.logspace(np.log10(df.area_pct.min() * 0.9), np.log10(df.area_pct.max() * 1.1), 40)
ax1.hist(df.area_pct, bins=bins, color=SERIES, edgecolor=SURFACE, linewidth=1)
ax1.set_xscale("log")
log_ticks(ax1.xaxis, AREA_TICKS)
ax1.set(title="Box area", xlabel="% of frame area (log)", ylabel="detections")

ax2.scatter(df.width_px, df.height_px, s=18, color=SERIES, alpha=0.5, edgecolors=SURFACE, linewidths=1)
lim = [min(df.width_px.min(), df.height_px.min()) * 0.8, max(df.width_px.max(), df.height_px.max()) * 1.2]
ax2.plot(lim, lim, color=MUTED, linewidth=1)
ax2.annotate("square (width = height)", (lim[1], lim[1]), xytext=(-4, -4), textcoords="offset points",
             ha="right", va="top", color=INK_2, fontsize=9)
ax2.set(xscale="log", yscale="log", xlim=lim, ylim=lim, title="Box width vs height",
        xlabel="width px (log)", ylabel="height px (log)")
log_ticks(ax2.xaxis, PX_TICKS)
log_ticks(ax2.yaxis, PX_TICKS)
plt.tight_layout()
plt.show()
print("Smallest side (min of width, height) percentiles, px:",
      df[["width_px", "height_px"]].min(axis=1).quantile([0.01, 0.05, 0.1, 0.5]).round(1).to_dict())

## Box shape

Width / height ratio, log scale. A swimmer seen from the side is **wide** (> 1); a person standing on the deck
is **tall** (< 1). Dives, starts and turns can be briefly upright, so this is a weak signal.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.2))
bins = np.logspace(np.log10(df.aspect.min() * 0.9), np.log10(df.aspect.max() * 1.1), 40)
ax.hist(df.aspect, bins=bins, color=SERIES, edgecolor=SURFACE, linewidth=1)
ax.set_xscale("log")
log_ticks(ax.xaxis, ASPECT_TICKS)
ax.axvline(1, color=MUTED, linewidth=1)
ax.annotate("tall  ←", (1, 1), xycoords=("data", "axes fraction"), xytext=(-4, -2), textcoords="offset points",
            ha="right", va="top", color=INK_2, fontsize=9)
ax.annotate("→  wide", (1, 1), xycoords=("data", "axes fraction"), xytext=(4, -2), textcoords="offset points",
            ha="left", va="top", color=INK_2, fontsize=9)
ax.set(title="Box shape (width / height)", xlabel="width / height (log)", ylabel="detections")
plt.show()
print(f"Tall boxes (width/height < 1): {(df.aspect < 1).sum()} of {len(df)}")

## Where the boxes are in the frame

Box centers over a frame from the middle of the video. Detections outside the pool (deck, stands, posters)
are candidates for a **pool-area** filter.

In [ ]:
def read_frame(frame_idx: int) -> np.ndarray:
    """One video frame as RGB."""
    cap = cv2.VideoCapture(str(VIDEO_PATH))
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()
    cap.release()
    if not ok:
        raise OSError(f"Cannot read frame {frame_idx} of {VIDEO_PATH}")
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

background = read_frame(n_frames // 2)
fig, ax = plt.subplots(figsize=(11, 11 * H / W))
ax.imshow(background, alpha=0.55)
ax.scatter(df.cx * W, df.cy * H, s=22, color=SERIES, alpha=0.7, edgecolors="white", linewidths=1)
ax.set(title=f"Box centers ({len(df)} detections) on frame {n_frames // 2}", xticks=[], yticks=[])
ax.grid(False)
plt.show()

## Score vs size

Do small boxes get low scores? If so, a size filter and a score filter remove the same boxes.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.scatter(df.area_pct, df.score, s=18, color=SERIES, alpha=0.5, edgecolors=SURFACE, linewidths=1)
ax.set(xscale="log", title="Score vs box area", xlabel="% of frame area (log)", ylabel="score")
log_ticks(ax.xaxis, AREA_TICKS)
plt.show()

## Over time

Gaps show where the swimmer was missed; spikes show extra persons (or false detections).

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
times = [f.timestamp for f in video_dets.frames]
ax1.step(times, counts, where="post", color=SERIES, linewidth=2)
ax1.set(title="Persons detected per frame", ylabel="persons")
ax1.yaxis.set_major_locator(MaxNLocator(integer=True))
ax2.scatter(df.time_s, df.score, s=12, color=SERIES, alpha=0.6, edgecolors="none")
ax2.set(title="Score of each detection over time", xlabel="time (s)", ylabel="score")
plt.tight_layout()
plt.show()

## Overlapping boxes

Detections that overlap another detection in the same frame (IoU = overlap, 0 = none, 1 = same box).
High overlap usually means the same person was detected twice (e.g. split by the water).

In [ ]:
multi = df[df.persons_in_frame > 1]
overlapping = df[df.max_iou > 0].sort_values("max_iou", ascending=False)
print(f"Detections in frames with more than one person: {len(multi)}")
print(f"  overlapping another box: {len(overlapping)}   with IoU > 0.5: {(df.max_iou > 0.5).sum()}")
overlapping[["frame", "score", "area_pct", "aspect", "max_iou"]].head(20).round(3)

## Look at the boxes

The numbers only help if we know what the odd boxes actually are. Below: crops of the most extreme detections
in each category. Use them to decide whether a filter would remove false detections or the real swimmer.

In [ ]:
def show_crops(rows: pd.DataFrame, title: str, n_cols: int = 6, pad: float = 0.25) -> None:
    """Crops of the given detections, with some context around each box."""
    if rows.empty:
        print(f"{title}: none")
        return
    n_rows = int(np.ceil(len(rows) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.2 * n_cols, 2.2 * n_rows), squeeze=False)
    fig.suptitle(title, x=0.01, ha="left", fontsize=11, fontweight="bold")
    for ax in axes.flat:
        ax.axis("off")
    for ax, (_, r) in zip(axes.flat, rows.iterrows()):
        frame = read_frame(int(r.frame))
        px, py = pad * (r.x2 - r.x1), pad * (r.y2 - r.y1)
        x1, y1 = int(max(r.x1 - px, 0)), int(max(r.y1 - py, 0))
        x2, y2 = int(min(r.x2 + px, W)), int(min(r.y2 + py, H))
        crop = frame[y1:y2, x1:x2].copy()
        thickness = max(2, round(max(crop.shape[:2]) / 120))  # stays visible after the crop is shrunk
        cv2.rectangle(crop, (int(r.x1) - x1, int(r.y1) - y1), (int(r.x2) - x1, int(r.y2) - y1), (255, 60, 60), thickness)
        ax.imshow(crop)
        ax.set_title(f"frame {int(r.frame)}  score {r.score:.2f}\narea {r.area_pct:.2f}%  w/h {r.aspect:.2f}",
                     fontsize=8, color=INK_2, fontweight="normal", loc="center")
    plt.tight_layout()
    plt.show()

N = 12
show_crops(df.nsmallest(N, "score"), "Lowest scores")
show_crops(df.nsmallest(N, "area_pct"), "Smallest boxes")
show_crops(df.nlargest(N, "area_pct"), "Largest boxes")
show_crops(df.nsmallest(N, "aspect"), "Tallest boxes (lowest width / height)")
show_crops(df[df.max_iou > 0].nlargest(N, "max_iou"), "Most overlapping boxes")

## Try filter parameters

Set a value to try a filter (`None` = off), then run the cell. It shows how many detections each filter would
remove on its own, all together, and how many frames would lose **all** their detections
(a warning sign that the real swimmer is being removed).

In [ ]:
MIN_SCORE = None     # e.g. 0.6
MIN_AREA_PCT = None  # e.g. 0.05  (% of frame area)
MAX_AREA_PCT = None  # e.g. 20
MIN_SIDE_PX = None   # e.g. 15  (smaller of width and height)
MIN_ASPECT = None    # e.g. 0.5  (drops very tall boxes)
MAX_ASPECT = None    # e.g. 10   (drops very flat boxes)

filters = {
    "score < MIN_SCORE": None if MIN_SCORE is None else df.score < MIN_SCORE,
    "area < MIN_AREA_PCT": None if MIN_AREA_PCT is None else df.area_pct < MIN_AREA_PCT,
    "area > MAX_AREA_PCT": None if MAX_AREA_PCT is None else df.area_pct > MAX_AREA_PCT,
    "side < MIN_SIDE_PX": None if MIN_SIDE_PX is None else df[["width_px", "height_px"]].min(axis=1) < MIN_SIDE_PX,
    "aspect < MIN_ASPECT": None if MIN_ASPECT is None else df.aspect < MIN_ASPECT,
    "aspect > MAX_ASPECT": None if MAX_ASPECT is None else df.aspect > MAX_ASPECT,
}
active = {name: mask for name, mask in filters.items() if mask is not None}
if not active:
    print("No filter set: set one of the values above and run this cell again.")
else:
    removed = np.logical_or.reduce(list(active.values()))
    frames_before = set(df.frame)
    frames_after = set(df.frame[~removed])
    summary = pd.DataFrame({name: {"removed": int(mask.sum()), "% of detections": round(100 * mask.mean(), 1)}
                            for name, mask in active.items()}).T
    summary.loc["ALL TOGETHER"] = [int(removed.sum()), round(100 * removed.mean(), 1)]
    summary["removed"] = summary["removed"].astype(int)
    print(summary.to_string())
    print(f"\nFrames with a person: {len(frames_before)} -> {len(frames_after)} "
          f"({len(frames_before - frames_after)} frames would lose all their detections)")
    show_crops(df[removed].sample(min(12, int(removed.sum())), random_state=0), "Sample of removed detections")